In [8]:
## Import required libraries
from transformers import pipeline
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load the feature-extraction pipeline with BERT
# We specify framework='tf' to use TensorFlow
print("Loading BERT pipeline...")
extractor = pipeline('feature-extraction', model='bert-base-uncased', framework='tf')
print("Pipeline loaded successfully!\n")

# 2. Define 10 sentence pairs (5 original + 5 new)
sentence_pairs = [
    # Original Pairs
    ("How do I learn Python?", "What is the best way to study Python?"),
    ("What is AI?", "How to cook pasta?"),
    ("How do I bake a chocolate cake?", "Give me a chocolate cake recipe."),
    ("How can I improve my coding skills?", "Tips for becoming better at programming."),
    ("Where can I buy cheap laptops?", "Best sites to find affordable computers."),

    # 5 New Pairs
    ("The weather is incredibly beautiful today.", "It is a lovely sunny day outside."),
    ("I need to buy a new car.", "My automobile requires an oil change immediately."),
    ("I love listening to jazz music.", "Music featuring saxophones and smooth rhythms is my favorite."),
    ("What time is the football match?", "When does the soccer game start today?"),
    ("The cat slept peacefully on the mat.", "Quantum physics is a highly difficult subject to grasp.")
]

# Ground truth similarity labels: 1 = similar, 0 = not similar
labels = [1, 0, 1, 1, 1, 1, 0, 1, 1, 0]

# 3. Function to get the BERT [CLS] embedding for a sentence
def get_sentence_embedding(sentence):
    """
    Uses the pipeline to extract the contextualized embedding of the [CLS] token.
    The pipeline returns a list of shape (1, sequence_length, hidden_size).
    """
    # Get all token embeddings
    embeddings = extractor(sentence)

    # The [CLS] token is the first element (index 0) of the sequence
    # embeddings[0] is the result for the first (and only) sentence
    cls_embedding = np.array(embeddings[0][0])

    return cls_embedding.reshape(1, -1)

# 4. Calculate cosine similarity and evaluate predictions
print("Evaluating sentence pairs...\n")
predictions = []

for i, (sent1, sent2) in enumerate(sentence_pairs):
    # Generate embeddings for both sentences
    emb1 = get_sentence_embedding(sent1)
    emb2 = get_sentence_embedding(sent2)

    # Calculate cosine similarity
    sim_score = cosine_similarity(emb1, emb2)[0][0]

    # Apply the 0.7 threshold
    pred = 1 if sim_score > 0.7 else 0
    predictions.append(pred)

    # Output results for the pair
    print(f"Pair {i+1}:")
    print(f"  S1: '{sent1}'")
    print(f"  S2: '{sent2}'")
    print(f"  Similarity: {sim_score:.4f} → Predicted: {pred} (Actual: {labels[i]})\n")

# 5. Evaluate overall accuracy
correct = sum(1 for p, l in zip(predictions, labels) if p == l)
total = len(labels)
accuracy = correct / total

print("-" * 40)
print(f"Final Model Accuracy: {accuracy:.2%}")
print("-" * 40)

Loading BERT pipeline...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pipeline loaded successfully!

Evaluating sentence pairs...

Pair 1:
  S1: 'How do I learn Python?'
  S2: 'What is the best way to study Python?'
  Similarity: 0.9743 → Predicted: 1 (Actual: 1)

Pair 2:
  S1: 'What is AI?'
  S2: 'How to cook pasta?'
  Similarity: 0.9033 → Predicted: 1 (Actual: 0)

Pair 3:
  S1: 'How do I bake a chocolate cake?'
  S2: 'Give me a chocolate cake recipe.'
  Similarity: 0.8938 → Predicted: 1 (Actual: 1)

Pair 4:
  S1: 'How can I improve my coding skills?'
  S2: 'Tips for becoming better at programming.'
  Similarity: 0.8633 → Predicted: 1 (Actual: 1)

Pair 5:
  S1: 'Where can I buy cheap laptops?'
  S2: 'Best sites to find affordable computers.'
  Similarity: 0.8750 → Predicted: 1 (Actual: 1)

Pair 6:
  S1: 'The weather is incredibly beautiful today.'
  S2: 'It is a lovely sunny day outside.'
  Similarity: 0.9424 → Predicted: 1 (Actual: 1)

Pair 7:
  S1: 'I need to buy a new car.'
  S2: 'My automobile requires an oil change immediately.'
  Similarity: 0.890

### Assignment Questions

**1. How does BERT differ from traditional NLP approaches like Bag of Words or TF-IDF?**
Traditional methods like Bag of Words or TF-IDF are frequency-based and treat words as independent tokens, losing the order and context of words. BERT (Bidirectional Encoder Representations from Transformers) is context-dependent, meaning the representation of a word changes based on the words surrounding it.

**2. What is the role of the encoder in the BERT model, and how is it used in this assignment?**
BERT is essentially a stack of Transformer Encoders. The encoder's role is to process the input sequence and represent each token as a high-dimensional vector that captures its relationship with every other token in the sequence. In this assignment, we use the encoder to generate dense vector representations (embeddings) of our sentences.

**3. What are contextual embeddings? How are they generated and used in this code?**
Contextual embeddings are word representations where the same word has different vectors depending on its context (e.g., 'bank' in 'river bank' vs 'bank account'). They are generated by the self-attention mechanism within BERT's layers. In this code, we extract the `last_hidden_state` from the model output to get these vectors.

**4. Why is the [CLS] token used for sentence similarity in this code?**
The `[CLS]` (Classification) token is a special token added to the beginning of every sequence. During pre-training, its embedding is intended to aggregate information from the entire sequence, making it a common choice for a fixed-size representation of a whole sentence.

**5. What is cosine similarity, and why is it useful in comparing embeddings?**
Cosine similarity measures the cosine of the angle between two vectors. It is useful for embeddings because it focuses on the orientation (semantic direction) of the vectors rather than their magnitude, allowing us to quantify how 'close' two concepts are in the high-dimensional vector space.